# Leaf-parallel virtual-loss MCTS — Colab T4 benchmark

Runs two benchmarks against the multi-threaded `BatchedMCTS` implementation:

1. **Per-move microbenchmark** (`benchmark_mcts.py`) — isolates the CPU↔GPU communication-overhead reduction across a sweep of (workers, batch size).
2. **Training-workload benchmark** (`benchmark_training.py`) — times `Coach.executeEpisode()` under single-threaded MCTS vs multi-threaded BatchedMCTS to give the realistic training-time speedup.

**Before running:** `Runtime → Change runtime type → GPU` (T4 is fine). Then run the cells in order.

In [1]:
# Cell 1: confirm GPU is attached.
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

Thu May  7 18:16:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 2: clone the repo and install missing deps.
# Replace REPO_URL with your fork's URL (or upload the repo as a zip and unzip in /content).
REPO_URL = 'https://github.com/YOUR_USERNAME/alpha-zero-general-forked-master.git'
REPO_DIR = '/content/azg'

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}


In [2]:
!pip install -q coloredlogs tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.8 MB/s eta 0:00:00


## 1. Per-move microbenchmark

Sweeps `numParallelSims` and `evalBatchSize`. Reports wall-clock per `getActionProb`, GPU-call count, mean batch size, lock wait %, and virtual-loss collisions. The interesting columns are **GPU calls** (collapses from `numMCTSSims` to `numMCTSSims / mean_batch_size`) and **wall-clock speedup**.

In [3]:
!python3 benchmark_mcts.py --sims 50 --workers 1,2,4,8,16 --batches 1,4,8,16 --repeat 3

python3: can't open file '/content/benchmark_mcts.py': [Errno 2] No such file or directory


## 2. Training-workload benchmark

Times `Coach.executeEpisode()` repeated `--episodes` times under single-threaded MCTS vs multi-threaded `BatchedMCTS`. This is the realistic training-time speedup — the number that informs whether the C/C++ port is worth the engineering cost.

Expected on T4: ~3–6× wall-clock speedup at `--workers 8 --batch 8`.

In [ ]:
!python3 benchmark_training.py --episodes 10 --sims 25 --workers 8 --batch 8

## 3. (Optional) Longer training run for tighter measurement variance

In [ ]:
!python3 benchmark_training.py --episodes 30 --sims 25 --workers 16 --batch 16